# Imports

In [1]:
import cda2
import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
import json

from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# Connect to Spark

In [2]:
api = cda2.Api()

Set configuration parameters to better optimize queries.

In [3]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

Start Spark and specify number of cpus to use. 400 is quite high, but we'll be running 1 year at a time and want to have it done in just a few minutes.

In [4]:
api.start_spark(n_executors=400, config=config)

https://artifacts.mitre.org/artifactory/java-libs-release added as a remote repository with the name: repo-1
https://dali.mitre.org/nexus/content/repositories/mitre-caasd-releases added as a remote repository with the name: repo-2
https://dali.mitre.org/nexus/content/repositories/external-releases added as a remote repository with the name: repo-3


:: loading settings :: url = jar:file:/devel/data_access/software/tdp-jupyter/poetry/cache/virtualenvs/python39-QwwvzYkJ-py3.9/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/rchong/.ivy2/cache
The jars for the packages stored in: /home/rchong/.ivy2/jars
org.mitre.spark#spark-geo_spark3.5_2.12 added as a dependency
org.apache.spark#spark-avro_2.12 added as a dependency
graphframes#graphframes added as a dependency
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
com.oracle.database.jdbc#ojdbc8 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6244a0e4-409c-46a4-a0b8-c9334f4f601a;1.0
	confs: [default]
	found org.mitre.spark#spark-geo_spark3.5_2.12;0.2.0 in repo-1
	found org.mitre.spark#spark-geo-core_2.12;0.2.0 in repo-1
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found net.sf.geographiclib#GeographicLib-Java;2.0 in central
	found org.ejml#ejml-core;0.43.1 in central
	found org.ejml#ejml-ddense;0.43.1 in central
	found com.esri.geometry#esri-geometry-api;2.2.4 in central
	found com.fasterxml.jackson.core#jackson-core;2.9.6 in central
	fou

Function to convert Unix timestamp (milliseconds from 1970) to YYYYMMDD string.

In [5]:
@F.udf("string")
def to_date(ts):
    return datetime.datetime.utcfromtimestamp(ts / 1000).strftime("%Y%m%d")

In [6]:
year0 = "2025"
year1 = str(int(year0) + 1)

In [7]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 +"-01-01"}

In [8]:
df_airports_raw = (
    api.dataframe("ArincAirport", **dates, metadata=True)
    .select(
        F.col("identification.name").alias("icao_code"),
        F.col("identification.icao_region").alias("icao_region"),
       "iata_code",
       "full_name",
        "latitude",
        "longitude",
        "elevation",
        F.col("magnetic_variation.modeled").alias("magnetic_variation"),
        F.col("metadata.effective_end_date").alias("end_date"),
    )
    .withColumn("full_name", F.regexp_replace("full_name", ",", ""))
#    .filter(F.col("usage") == "PUBLIC")
#    .sort(F.desc("end_date"))
    .orderBy("icao_code", "icao_region")
)

#df_airports_raw.show()

Multiple versions found: 3.1.71, 3.1.73, 3.1.74, 3.1.75, 3.1.76, 3.1.77, 3.1.79, 3.1.80
                                                                                

In [9]:
#df_airports_raw.count()

In [10]:
window = Window.partitionBy("icao_code").orderBy(col("end_date").desc())

df_airports = (df_airports_raw
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row")
)
 
#df_airports.show()

In [11]:
#df_airports.count()

In [12]:
(
    df_airports
    .write.option("header", True)
    .csv("CRAFT/" + year0 + "/airports", compression="None", mode="overwrite")
)